## Prerequisites

Before understanding XGBoost for classification, the following concepts should be clear:

**1) XGBoost Regression**

- Works using additive tree building  
- Each tree improves previous predictions  
- Optimizes an objective function:
  - Loss (e.g., Mean Squared Error)
  - Regularization  
- Understanding of leaf weights and tree structure  

**2) Gradient Boosting in Classification**

- Sequential learning where each model corrects previous errors  
- Uses loss functions like log loss (binary cross-entropy)  
- Learns using gradients instead of direct residuals  
- Converts classification into an optimization problem  

**3) Log-Odds (Logit)**

- Odds:
  $$
  \frac{P}{1 - P}
  $$

- Log-odds:
  $$
  \log\left(\frac{P}{1 - P}\right)
  $$

- XGBoost predicts log-odds instead of probability  

- Converted to probability using sigmoid:
  $$
  P = \frac{1}{1 + e^{-z}}
  $$

--------

## Disclaimer

- XGBoost is an improved version of gradient boosting with better performance and speed  
- It involves heavy mathematical calculations  

- The focus should be on understanding the concepts rather than manual calculations  
- In real-world applications, libraries handle the computations  
- Deep manual derivations are mainly important for research, not for practical usage  

-----
----

### Dataset

| CGPA | Placed? |
|------|--------|
| 5.70 | 0      |
| 6.25 | 1      |
| 7.10 | 0      |
| 8.15 | 1      |
| 9.60 | 1      |

### Overview

XGBoost can be understood as an optimized version of gradient boosting, designed to improve both performance and speed.

The goal is to build a model that takes a student's CGPA as input and predicts whether the student will be placed or not.

The process starts with a base model. In classification, this base prediction is initialized using log-odds instead of direct probabilities. This provides a stable starting point for optimization.

$$
\text{log-odds} = \log\left(\frac{P}{1 - P}\right)
$$

After the initial prediction, we compute the errors. Instead of directly correcting predictions, XGBoost fits a decision tree on these errors (more precisely, gradients). This becomes the second stage of the model.

Each new tree is trained sequentially to correct the mistakes made by the previous model. The final prediction is a combination of all trees, where each tree's contribution is scaled by a learning rate.

$$
\hat{y} = \sum_{m=1}^{M} \eta \cdot f_m(x)
$$

where:

$$
f_m(x) \text{ = prediction from the } m\text{-th tree}
$$

$$
\eta \text{ = learning rate}
$$

A key difference from standard gradient boosting lies in how trees are built. Traditional gradient boosting uses decision trees based on criteria like Gini or entropy. In contrast, XGBoost uses trees built using similarity scores derived from gradients and second-order gradients.

This allows better split decisions and more optimized learning.

In essence, the model is an ensemble of multiple decision trees, each trained to correct previous errors, with a more mathematically optimized approach to splitting and prediction.

---
---

## Step by Step Solution

| CGPA | Placed? |
|------|--------|
| 5.70 | 0      |
| 6.25 | 1      |
| 7.10 | 0      |
| 8.15 | 1      |
| 9.60 | 1      |

### Stage 1 — Base Estimator

In regression, the base prediction starts with the mean of the target variable.  
In classification (XGBoost), we start with **log-odds**.

$$
\text{log-odds} = \log\left(\frac{p}{1 - p}\right)
$$

From the dataset:

$$
p = \frac{3}{5}
$$

$$
\text{log-odds} = \log\left(\frac{3}{2}\right) \approx 0.405
$$

This becomes the **initial prediction for every data point**, regardless of CGPA.

$$
\hat{y}^{(0)} = 0.405
$$

To interpret this, we convert log-odds into probability using the sigmoid function:

$$
P = \frac{e^{z}}{1 + e^{z}}
$$

$$
P = \frac{e^{0.405}}{1 + e^{0.405}} \approx 0.60
$$

So the base model predicts the same probability for all samples:

$$
\text{Pred}_1 = [0.6, 0.6, 0.6, 0.6, 0.6]
$$

At this stage, the model does not consider CGPA — it only reflects the overall distribution of the target.

| CGPA | Placed? | pred₁ (log-odds) | pred₁ (prob) |
|------|--------|------------------|--------------|
| 5.70 | 0      | 0.405            | 0.6          |
| 6.25 | 1      | 0.405            | 0.6          |
| 7.10 | 0      | 0.405            | 0.6          |
| 8.15 | 1      | 0.405            | 0.6          |
| 9.60 | 1      | 0.405            | 0.6          |

---

### Residuals (Stage 1)

Now we compute the error of the base model predictions.  
This error is called the **residual**.

$$
\text{Residual} = y_{\text{true}} - y_{\text{pred}}
$$

Since:

$$
\text{Pred}_1 = [0.6, 0.6, 0.6, 0.6, 0.6]
$$

and

$$
y = [0, 1, 0, 1, 1]
$$

Residuals:

$$
\text{Res}_1 = [-0.6, 0.4, -0.6, 0.4, 0.4]
$$

These residuals represent how much the model is wrong for each data point.

Stage 1 ends here. The next step is to train a decision tree on these residuals.

----
---

### Stage 2 — Error Prediction (Tree on Residuals)

In this stage, we train a new model to predict the errors made by the previous model.

Input feature: CGPA  
Target: res₁  

| CGPA | res₁  |
|------|-------|
| 5.70 | -0.6  |
| 6.25 | 0.4   |
| 7.10 | -0.6  |
| 8.15 | 0.4   |
| 9.60 | 0.4   |

We begin by considering all data points in a single leaf node:

$$
[-0.6,\; 0.4,\; -0.6,\; 0.4,\; 0.4]
$$

Now we calculate the **Similarity Score (SS)**.

$$
SS = \frac{\left(\sum r_i\right)^2}{\sum p_i(1 - p_i) + \lambda}
$$

where:

$$
r_i = \text{residuals}, \quad p_i = \text{previous probabilities}
$$

Since all previous probabilities are:

$$
p_i = 0.6
$$

Step-by-step:

$$
\sum r_i = (-0.6 + 0.4 - 0.6 + 0.4 + 0.4) = 0
$$

$$
\sum p_i(1 - p_i) = 5 \times (0.6 \times 0.4) = 5 \times 0.24 = 1.2
$$

$$
SS = \frac{0^2}{1.2 + 0} = 0
$$

This means the current node has **no predictive power**.

Now we try splitting the data based on CGPA.

Goal: find a split that **maximizes similarity score (SS)**.

Each possible split:
- divides data into left and right nodes  
- compute SS for both nodes  
- choose the split with maximum gain  

This is how XGBoost decides the best split, unlike traditional trees that use Gini or entropy.

### Splitting Criteria

We sort the CGPA values and take the average of consecutive values to generate possible splits:

$$
5.97,\; 6.67,\; 7.62,\; 8.87
$$

# Now we evaluate the first split:

$$
\text{Split: CGPA} < 5.97
$$

Left node (Yes):

$$
[-0.6]
$$

Right node (No):

$$
[0.4,\; -0.6,\; 0.4,\; 0.4]
$$

### Similarity Score Calculation

Formula:

$$
SS = \frac{\left(\sum r_i\right)^2}{\sum p_i(1 - p_i) + \lambda}
$$

Here:

$$
p_i = 0.6,\quad \lambda = 0
$$

#### Left Node

$$
\sum r_i = -0.6
$$

$$
\sum p_i(1 - p_i) = 0.6 \times 0.4 = 0.24
$$

$$
SS_{\text{left}} = \frac{(-0.6)^2}{0.24} = \frac{0.36}{0.24} = 1.5
$$

#### Right Node

$$
\sum r_i = (0.4 - 0.6 + 0.4 + 0.4) = 0.6
$$

$$
\sum p_i(1 - p_i) = 4 \times 0.24 = 0.96
$$

$$
SS_{\text{right}} = \frac{(0.6)^2}{0.96} = \frac{0.36}{0.96} = 0.375
$$

### Total Similarity After Split

$$
SS_{\text{total}} = 1.5 + 0.375 = 1.875
$$

### Gain

Initial SS was:

$$
SS_{\text{parent}} = 0
$$

$$
\text{Gain} = 1.875 - 0 = 1.875
$$

This shows that the split improves the model significantly. Remaining splits will be evaluated similarly, and the split with the highest gain will be selected.

---

## note

why this is not left node [-0.6, -0.6]
This is actually important:

- XGBoost allows single-point leaf nodes
- But such splits may not always be optimal
- That’s why we compute gain for all splits and pick the best one
---

### Split 2 Evaluation

$$
\text{Split: CGPA} < 6.67
$$

Left node:

$$
[-0.6,\; 0.4]
$$

Right node:

$$
[-0.6,\; 0.4,\; 0.4]
$$

### Similarity Score

$$
SS = \frac{\left(\sum r_i\right)^2}{\sum p_i(1 - p_i)}
$$

Since:

$$
p_i = 0.6 \Rightarrow p_i(1-p_i) = 0.24
$$

---

#### Left Node

$$
\sum r_i = (-0.6 + 0.4) = -0.2
$$

$$
\sum p_i(1-p_i) = 2 \times 0.24 = 0.48
$$

$$
SS_{\text{left}} = \frac{(-0.2)^2}{0.48} = \frac{0.04}{0.48} \approx 0.083
$$

---

#### Right Node

$$
\sum r_i = (-0.6 + 0.4 + 0.4) = 0.2
$$

$$
\sum p_i(1-p_i) = 3 \times 0.24 = 0.72
$$

$$
SS_{\text{right}} = \frac{(0.2)^2}{0.72} = \frac{0.04}{0.72} \approx 0.055
$$

---

### Gain

$$
\text{Gain} = SS_{\text{left}} + SS_{\text{right}} - SS_{\text{parent}}
$$

$$
\text{Gain} = 0.083 + 0.055 - 0 \approx 0.138
$$

---
---

### Split 3 Evaluation

$$
\text{Split: CGPA} < 7.62
$$

Left node:

$$
[-0.6,\; 0.4,\; -0.6]
$$

Right node:

$$
[0.4,\; 0.4]
$$

### Similarity Score

$$
SS = \frac{\left(\sum r_i\right)^2}{\sum p_i(1 - p_i)}
$$

Since:

$$
p_i = 0.6 \Rightarrow p_i(1-p_i) = 0.24
$$

---

#### Left Node

$$
\sum r_i = (-0.6 + 0.4 - 0.6) = -0.8
$$

$$
\sum p_i(1-p_i) = 3 \times 0.24 = 0.72
$$

$$
SS_{\text{left}} = \frac{(-0.8)^2}{0.72} = \frac{0.64}{0.72} \approx 0.89
$$

---

#### Right Node

$$
\sum r_i = (0.4 + 0.4) = 0.8
$$

$$
\sum p_i(1-p_i) = 2 \times 0.24 = 0.48
$$

$$
SS_{\text{right}} = \frac{(0.8)^2}{0.48} = \frac{0.64}{0.48} \approx 1.33
$$

---

### Gain

$$
\text{Gain} = SS_{\text{left}} + SS_{\text{right}} - SS_{\text{parent}}
$$

$$
\text{Gain} = 0.89 + 1.33 - 0 \approx 2.22
$$

---
---
### Split 4 Evaluation

$$
\text{Split: CGPA} < 8.87
$$

Left node:

$$
[-0.6,\; 0.4,\; -0.6,\; 0.4]
$$

Right node:

$$
[0.4]
$$

### Similarity Score

$$
SS = \frac{\left(\sum r_i\right)^2}{\sum p_i(1 - p_i)}
$$

Since:

$$
p_i = 0.6 \Rightarrow p_i(1-p_i) = 0.24
$$

---

#### Left Node

$$
\sum r_i = (-0.6 + 0.4 - 0.6 + 0.4) = -0.4
$$

$$
\sum p_i(1-p_i) = 4 \times 0.24 = 0.96
$$

$$
SS_{\text{left}} = \frac{(-0.4)^2}{0.96} = \frac{0.16}{0.96} \approx 0.167
$$

---

#### Right Node

$$
\sum r_i = 0.4
$$

$$
\sum p_i(1-p_i) = 1 \times 0.24 = 0.24
$$

$$
SS_{\text{right}} = \frac{(0.4)^2}{0.24} = \frac{0.16}{0.24} \approx 0.667
$$

---

### Gain

$$
\text{Gain} = SS_{\text{left}} + SS_{\text{right}} - SS_{\text{parent}}
$$

$$
\text{Gain} = 0.167 + 0.667 - 0 \approx 0.834
$$

### Stage 2 Model

The best split is obtained at:

$$
\text{CGPA} < 7.62
$$

since it gives the maximum gain:

$$
\text{Gain} \approx 2.22
$$

So the decision tree for Stage 2 becomes:

- If $$ \text{CGPA} < 7.62 $$  
  $$ [-0.6,\; 0.4,\; -0.6] $$  

- Else  
  $$ [0.4,\; 0.4] $$  

Since we are using a shallow tree, we stop splitting here.

This tree is trained on residuals, so instead of predicting class labels, it predicts **error corrections** for the base model.

This becomes the **Stage 2 model**, which will be added to the base model (with a learning rate) to improve predictions.

----
----

### Leaf Node Output Calculation

Now we compute the output value for each leaf node.

Formula:

$$
\text{Output} = \frac{\sum r_i}{\sum p_i(1 - p_i) + \lambda}
$$

Here:

$$
\lambda = 0,\quad p_i = 0.6 \Rightarrow p_i(1-p_i) = 0.24
$$

---

#### Left Leaf

$$
[-0.6,\; 0.4,\; -0.6]
$$

$$
\sum r_i = (-0.6 + 0.4 - 0.6) = -0.8
$$

$$
\sum p_i(1-p_i) = 3 \times 0.24 = 0.72
$$

$$
\text{Output}_{\text{left}} = \frac{-0.8}{0.72} \approx -1.11
$$

---

#### Right Leaf

$$
[0.4,\; 0.4]
$$

$$
\sum r_i = 0.8
$$

$$
\sum p_i(1-p_i) = 2 \times 0.24 = 0.48
$$

$$
\text{Output}_{\text{right}} = \frac{0.8}{0.48} \approx 1.67
$$

---

These values are the **predictions of the Stage 2 tree**.

They will be added to the base model predictions (scaled by learning rate) in the next step.

----
---

## Stage 2 — Prediction Update

Now we update the predictions using the Stage 2 model.

Formula:

$$
\hat{y} = m_1 + \eta \cdot m_2
$$

where:

$$
m_1 = 0.405,\quad \eta = 0.3
$$

Decision rule from Stage 2 tree:

$$
\text{If } \text{CGPA} < 7.62 \Rightarrow m_2 = -1.11
$$

$$
\text{Else } m_2 = 1.67
$$

---

### Updated Log-Odds

For CGPA < 7.62:

$$
\hat{y} = 0.405 + 0.3 \times (-1.11)
$$

$$
\hat{y} = 0.405 - 0.333 = 0.072
$$

---

For CGPA ≥ 7.62:

$$
\hat{y} = 0.405 + 0.3 \times 1.67
$$

$$
\hat{y} = 0.405 + 0.501 = 0.906
$$

---

### Final Predictions (log-odds)

$$
[0.072,\; 0.072,\; 0.072,\; 0.906,\; 0.906]
$$

### Updated Predictions (After Stage 2)

Probability is calculated using:

$$
P = \frac{e^{z}}{1 + e^{z}}
$$

| CGPA | Placed? | pred₁ (log-odds) | pred₁ (prob) | res₁  | pred₂ (log-odds) | pred₂ (prob) | res₂  |
|------|--------|------------------|--------------|-------|------------------|--------------|-------|
| 5.70 | 0      | 0.405            | 0.6          | -0.6  | 0.072            | 0.518        | -0.518 |
| 6.25 | 1      | 0.405            | 0.6          | 0.4   | 0.072            | 0.518        | 0.482  |
| 7.10 | 0      | 0.405            | 0.6          | -0.6  | 0.072            | 0.518        | -0.518 |
| 8.15 | 1      | 0.405            | 0.6          | 0.4   | 0.906            | 0.712        | 0.288  |
| 9.60 | 1      | 0.405            | 0.6          | 0.4   | 0.906            | 0.712        | 0.288  |

----
----

## Stage 3

Now we repeat the same process using the new residuals.

Input:

$$
\text{CGPA} \;\;|\;\; \text{res}_2
$$

A new decision tree (m₃) is trained on these residuals.

Steps:
- Try all possible splits on CGPA  
- Compute similarity score for each split  
- Select the split with maximum gain  
- Compute leaf outputs using:

$$
\text{Output} = \frac{\sum r_i}{\sum p_i(1 - p_i) + \lambda}
$$

---

### Updated Model

Now the model becomes:

$$
\hat{y} = m_1 + \eta \cdot m_2 + \eta \cdot m_3
$$

$$
\hat{y} = 0.405 + 0.3 \cdot m_2 + 0.3 \cdot m_3
$$

---

### Prediction Update

- Convert updated log-odds to probability:

$$
P = \frac{e^{z}}{1 + e^{z}}
$$

- Compute new residuals:

$$
\text{res}_3 = y - P
$$

---

This process continues iteratively:
- Each new tree learns from previous errors  
- Predictions get refined step by step  
- Final model is a sum of all trees  

$$
\hat{y} = \sum_{m=1}^{M} \eta \cdot f_m(x)
$$

## Final Understanding

This is the complete working flow of XGBoost for classification.

It closely follows the same idea as gradient boosting for classification:
- Start with a base prediction (log-odds)  
- Compute residuals (errors)  
- Train trees sequentially on these errors  
- Update predictions step by step  

The key differences lie in how XGBoost builds and optimizes trees:

- Instead of using Gini or entropy, it uses **Similarity Score (SS)**  
- Split decisions are based on **gain maximization**  
- It uses both **gradients and second-order information**  
- Leaf outputs are computed using a mathematically optimized formula  

$$
SS = \frac{\left(\sum r_i\right)^2}{\sum p_i(1 - p_i) + \lambda}
$$

$$
\text{Output} = \frac{\sum r_i}{\sum p_i(1 - p_i) + \lambda}
$$

The model grows as:

$$
\hat{y} = m_1 + \eta m_2 + \eta m_3 + \cdots
$$

At each stage:
- Convert log-odds to probability  
- Compute new residuals  
- Train the next tree  

This iterative refinement is what makes XGBoost powerful.

In the next notebook, we will go deeper into the **full mathematical derivation of XGBoost**, including:
- Taylor expansion  
- First and second order gradients  
- Exact gain formula derivation  